# Oracle Agentic Audit: pinned public saved-record reproduction

This notebook downloads fixed Atlas inputs and separate recorded research experiments,
refits CPU baselines, recomputes statistics and exports the revised paper's 18 numeric
tables and eight figures. It makes **no LLM requests**. Labels are protocol adjudications,
not independent truth. B2 uses two matched repeats; optional low-temperature experiments
were cancelled. A local run is labeled local; a fresh hosted Colab run is recorded separately.

Choose **Run all** in a fresh CPU runtime. Network access is needed for the public GitHub
checkout, approximately 188 MB of records and Python packages. Allow about 13 minutes
for the CPU analysis after installation; hardware and network speed vary.


In [ ]:
import os, sys, json, time, platform, subprocess, tempfile, pathlib, importlib.util
from datetime import datetime, timezone
from IPython.display import display, Image, Markdown
CODE_REPOSITORY = "https://github.com/virusLuke3/Oracle-Agentic-Audit.git"
CODE_REVISION = "33ec7020e97fc6b8dd0005415d3184ce889d3e28"
STARTED_UTC = datetime.now(timezone.utc).isoformat()
START = time.monotonic()
WORK = pathlib.Path(tempfile.mkdtemp(prefix="oracle-public-replay-"))
ROOT = WORK / "Oracle-Agentic-Audit"
ENV = dict(os.environ, MPLBACKEND="Agg", OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", UV_HTTP_TIMEOUT="180", UV_HTTP_RETRIES="5", UV_CONCURRENT_DOWNLOADS="4", PIP_PROGRESS_BAR="off")
def run(args, cwd=None):
    result = subprocess.run([str(x) for x in args], cwd=cwd or WORK, env=ENV,
                            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    result.check_returncode()
    return result.stdout
run(["git", "clone", "--quiet", "--no-checkout", CODE_REPOSITORY, ROOT])
run(["git", "checkout", "--quiet", "--detach", CODE_REVISION], ROOT)
assert run(["git", "rev-parse", "HEAD"], ROOT).strip() == CODE_REVISION
print("Fresh working directory:", WORK)


## Isolated Python environment

Python 3.12.7 and all scientific package versions are pinned. A private bootstrap
installs uv 0.7.16; uv selects/downloads Python 3.12.7 even if the notebook kernel
uses another Python version. No system package or GPU service is required.


In [ ]:
BOOT = WORK / "bootstrap"
run([sys.executable, "-m", "venv", BOOT])
BPY = BOOT / "bin/python"
run([BPY, "-m", "pip", "install", "--disable-pip-version-check", "uv==0.7.16"])
UV = BOOT / "bin/uv"
run([UV, "venv", "--python", "3.12.7", ROOT / ".venv"])
PY = ROOT / ".venv/bin/python"
run([PY, "-m", "ensurepip", "--upgrade"])
run([PY, "-m", "pip", "install", "--disable-pip-version-check", "--timeout", "180", "--retries", "5", "-r", ROOT / "requirements.lock"])
run([PY, "-m", "pip", "check"])
SCIENTIFIC_PYTHON = run([PY, "--version"]).strip()
print(SCIENTIFIC_PYTHON)


## Fetch and verify the fixed public records

Both archives are pinned to the dataset commit recorded in the code release.
SHA-256 checks cover the archives and every file. This does not certify the full
Atlas resource or an independently reviewed model checkpoint.


In [ ]:
PIN = json.loads((ROOT / "manifests/public-release.json").read_text())
print(json.dumps(PIN, indent=2))
run([PY, "-m", "experiments.download"], ROOT)
run([PY, "-m", "unittest", "discover", "-s", "tests", "-v"], ROOT)
run([PY, "-m", "oracle_audit", "experiment-check"], ROOT)


## Recompute the research

This cell verifies original response records, refits B0/B0+ on training cases,
calibrates only on validation cases, reconstructs the cohort/features, evaluates
policies and safeguards, computes 2,000-resample case-bootstrap intervals, and
regenerates the tables/figures. Saved responses are not fresh model generations.


In [ ]:
OUT = ROOT / "outputs/notebook_replay"
run([PY, "-m", "analysis.replay", "--retrain", "--out", OUT], ROOT)
run([PY, "-m", "analysis.verify_replay", "--reference", ROOT / "outputs/final", "--candidate", OUT], ROOT)
COMPARISON = json.loads((OUT / "comparison_receipt.json").read_text())
assert COMPARISON["status"] == "PASS"
assert COMPARISON["compared_tables"] == 73
assert len(COMPARISON["rendered_tables_and_prompts"]) == 19
assert len(COMPARISON["figures"]) == 8
print("PASS: 73 CSVs, 18 numeric tables plus exact prompts; eight regenerated figures.")


## Inspect results and limitations

Coverage/automation does not establish predictive superiority. Token workload is
not a measured dollar cost. Archived failed requests with unavailable usage are not
assigned zero cost. The following are actual figures generated by this run.


In [ ]:
display(Markdown((OUT / "RESULTS_zh.md").read_text()))
for name in ["workflow", "calibration", "citation_errors", "cost_performance", "guard_ablation", "economic_concentration", "policy_sensitivity", "tool_budget"]:
    display(Image(filename=str(OUT / "figures" / (name + ".png"))))


## Save the execution receipt and results

Download the receipt/result archive and save this executed notebook. Hosted-Colab
status is recorded only when this run is actually inside a Google Colab runtime.
The repository's outstanding arXiv/independent-review gates are not waived.


In [ ]:
import hashlib, tarfile
try:
    from google.colab import files as colab_files
    HOSTED = bool(os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_BACKEND_VERSION"))
except ImportError:
    colab_files = None
    HOSTED = False
RECEIPT = dict(status="PASS", execution_kind="HOSTED_GOOGLE_COLAB" if HOSTED else ("GITHUB_ACTIONS_FRESH_NOTEBOOK" if os.environ.get("GITHUB_ACTIONS")=="true" else "LOCAL_FRESH_NOTEBOOK"),
               hosted_colab="PASS" if HOSTED else "NOT_RUN", code_revision=CODE_REVISION,
               dataset_revision=PIN["dataset_revision"], started_utc=STARTED_UTC,
               completed_utc=datetime.now(timezone.utc).isoformat(), elapsed_seconds=time.monotonic()-START,
               kernel_python=platform.python_version(), scientific_python=SCIENTIFIC_PYTHON,
               platform=platform.platform(), llm_inference_requests=0, compared_csv_tables=73,
               rendered_numeric_tables=18, exact_prompt_listings=1, figures=8,
               comparison_sha256=hashlib.sha256((OUT/"comparison_receipt.json").read_bytes()).hexdigest(),
               clean_checkout_before_run=True)
(OUT / "notebook_execution_receipt.json").write_text(json.dumps(RECEIPT, indent=2)+"\n")
(OUT / "environment.freeze.txt").write_text(run([PY,"-m","pip","freeze"]))
print(json.dumps(RECEIPT, indent=2))
RESULT_ARCHIVE = WORK / "oracle-replay-results.tar.gz"
with tarfile.open(RESULT_ARCHIVE, "w:gz") as archive:
    archive.add(OUT, arcname="oracle-replay-results")
print("Saved:", RESULT_ARCHIVE)
if HOSTED:
    colab_files.download(str(RESULT_ARCHIVE))
